# Num2Text - LSTM + Greedy

- 숫자·영어·한국어 6방향 번역
- LSTM Encoder–Decoder / Greedy Decoding / Teacher Forcing
- BOS, EOS, PAD / CrossEntropyLoss
- **학습 설정:** Epoch 20, Batch 256, Embedding 128, Hidden 256, LR 0.001
- **데이터:** 두 파일의 공통 숫자 전체 사용 (`MAX_ROWS = None`)
- 숫자 입력은 문자열로 유지합니다.

> 기존 모델 파일은 설정이 다르므로 학습 후 새로 저장해야 합니다.


## 1. 준비 및 데이터 읽기

In [2]:
import random
from pathlib import Path
from urllib.request import urlretrieve

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# 데이터 파일이 없으면 GitHub에서 가져옵니다.
base = "https://raw.githubusercontent.com/YunOh21/2026-nlp/main/practice01/yun/"
for name in ["num2txt.txt", "num2kor.txt"]:
    if not Path(name).exists():
        try:
            urlretrieve(base + name, name)
        except Exception as e:
            print(name, "다운로드 실패:", e)

# 숫자와 읽기 문자열을 탭 기준으로 분리합니다.
def read_data(filename):
    data = {}
    with open(filename, encoding="utf-8-sig") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2 and parts[0] and parts[1]:
                data[parts[0].strip()] = parts[1].strip()
    return data

en = read_data("num2txt.txt")
ko = read_data("num2kor.txt")
keys = sorted(set(en) & set(ko))
assert keys, "두 파일에서 공통 숫자를 찾지 못했습니다. 파일 형식을 확인하세요."
print("공통 데이터 수:", len(keys))
print("예시:", keys[0], en[keys[0]], ko[keys[0]])

사용 장치: cpu
공통 데이터 수: 12738
예시: 0 zero 영


## 2. 6방향 데이터 및 단어 사전 만들기

영어는 **단어 단위**, 숫자와 한글은 **글자 단위**로 나눕니다. 같은 숫자의 번역이 학습과 시험에 동시에 들어가지 않도록 **숫자별로 먼저 분리**합니다.

In [3]:
# 학습 설정
MAX_ROWS = None
EPOCHS = 20
BATCH_SIZE = 256
EMBED_SIZE = 128
HIDDEN_SIZE = 256
LEARNING_RATE = 0.001

# 같은 숫자가 학습/시험에 중복되지 않도록 먼저 나눕니다.
random.shuffle(keys)
if MAX_ROWS is not None:
    keys = keys[:MAX_ROWS]
cut = int(len(keys) * 0.9)
train_keys, test_keys = keys[:cut], keys[cut:]

def make_pairs(numbers):
    result = []
    for number in numbers:
        values = {"num": number, "en": en[number], "ko": ko[number]}
        for source in values:
            for target in values:
                if source != target:
                    result.append((source, target, values[source], values[target]))
    return result

train_pairs = make_pairs(train_keys)
test_pairs = make_pairs(test_keys)

# 영어는 단어, 숫자와 한글은 문자 단위로 처리합니다.
def tokenize(text, lang):
    return text.split() if lang == "en" else list(text.replace(" ", ""))

vocab = {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3}
for source, target, x, y in train_pairs:
    for word in ["<to_" + target + ">"] + tokenize(x, source) + tokenize(y, target):
        if word not in vocab:
            vocab[word] = len(vocab)

print("학습 쌍:", len(train_pairs), "시험 쌍:", len(test_pairs), "어휘 수:", len(vocab))

학습 쌍: 68784 시험 쌍: 7644 어휘 수: 62


## 3. DataLoader

길이가 다른 문장은 `PAD`로 채웁니다. Encoder에는 실제 길이만 전달합니다.

In [4]:
# 배치마다 길이를 맞춰 PAD를 채웁니다.
def make_batch(batch):
    sources, targets, lengths = [], [], []
    for source, target, x, y in batch:
        x_words = ["<to_" + target + ">"] + tokenize(x, source) + ["<eos>"]
        y_words = ["<bos>"] + tokenize(y, target) + ["<eos>"]
        x_ids = [vocab.get(w, 3) for w in x_words]
        y_ids = [vocab.get(w, 3) for w in y_words]
        sources.append(torch.tensor(x_ids))
        targets.append(torch.tensor(y_ids))
        lengths.append(len(x_ids))
    x = pad_sequence(sources, batch_first=True, padding_value=0)
    y = pad_sequence(targets, batch_first=True, padding_value=0)
    return x.to(device), torch.tensor(lengths), y.to(device)

train_loader = DataLoader(train_pairs, batch_size=BATCH_SIZE, shuffle=True, collate_fn=make_batch)
test_loader = DataLoader(test_pairs, batch_size=BATCH_SIZE, collate_fn=make_batch)

## 4. LSTM Encoder와 Decoder

Encoder의 마지막 `(hidden, cell)`을 Decoder에 넘깁니다. 학습 중에는 정답 문장을 한 칸 앞당겨 Decoder에 입력합니다(Teacher Forcing).

In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x, lengths):
        x = self.embedding(x)
        # PAD를 제외한 실제 길이로 LSTM에 전달합니다.
        x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, state = self.lstm(x)
        return state

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, y, state):
        output, state = self.lstm(self.embedding(y), state)
        return self.fc(output), state

encoder = Encoder(len(vocab), EMBED_SIZE, HIDDEN_SIZE).to(device)
decoder = Decoder(len(vocab), EMBED_SIZE, HIDDEN_SIZE).to(device)

## 5. 모델 학습

Decoder 입력은 `BOS, y1, y2, ...`이고 정답은 `y1, y2, ..., EOS`입니다. `CrossEntropyLoss(ignore_index=0)`으로 PAD를 제외합니다.

> 전체 데이터를 학습하므로 GPU 사용을 권장합니다.


In [6]:
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    total_loss = 0
    for x, lengths, y in train_loader:
        optimizer.zero_grad()
        state = encoder(x, lengths)
        # Decoder 입력은 BOS부터, 정답은 다음 토큰부터 비교합니다.
        output, _ = decoder(y[:, :-1], state)
        loss = loss_fn(output.reshape(-1, len(vocab)), y[:, 1:].reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{EPOCHS} - Loss: {total_loss / len(train_loader):.4f}")

Epoch 1/20 - Loss: 1.1514
Epoch 2/20 - Loss: 0.1487
Epoch 3/20 - Loss: 0.0604
Epoch 4/20 - Loss: 0.0339
Epoch 5/20 - Loss: 0.0222
Epoch 6/20 - Loss: 0.0161
Epoch 7/20 - Loss: 0.0124
Epoch 8/20 - Loss: 0.0092
Epoch 9/20 - Loss: 0.0073
Epoch 10/20 - Loss: 0.0063
Epoch 11/20 - Loss: 0.0063
Epoch 12/20 - Loss: 0.0057
Epoch 13/20 - Loss: 0.0040
Epoch 14/20 - Loss: 0.0039
Epoch 15/20 - Loss: 0.0033
Epoch 16/20 - Loss: 0.0031
Epoch 17/20 - Loss: 0.0037
Epoch 18/20 - Loss: 0.0030
Epoch 19/20 - Loss: 0.0022
Epoch 20/20 - Loss: 0.0017


## 6. Greedy 번역 및 시험

이전 단계에서 확률이 가장 높은 토큰 하나를 고르고, `EOS`가 나올 때까지 반복합니다.

In [7]:
def translate(text, source, target):
    encoder.eval()
    decoder.eval()
    words = ["<to_" + target + ">"] + tokenize(text, source) + ["<eos>"]
    ids = [vocab.get(w, 3) for w in words]
    x = torch.tensor([ids], device=device)
    result = []
    reverse_vocab = {i: w for w, i in vocab.items()}

    with torch.no_grad():
        state = encoder(x, torch.tensor([len(ids)]))
        token = torch.tensor([[vocab["<bos>"]]], device=device)
        for _ in range(50):
            output, state = decoder(token, state)
            # 가장 확률이 높은 토큰을 하나씩 선택합니다.
            next_id = output[0, -1].argmax().item()
            if next_id == vocab["<eos>"]:
                break
            result.append(reverse_vocab.get(next_id, "<unk>"))
            token = torch.tensor([[next_id]], device=device)
    return " ".join(result) if target == "en" else "".join(result)

for source, target, x, y in test_pairs[:6]:
    print(f"{source} → {target}: {x}")
    print("정답:", y)
    print("예측:", translate(x, source, target))
    print()

num → en: 2157
정답: two thousand one hundred fifty seven
예측: two thousand one hundred fifty seven

num → ko: 2157
정답: 이천백오십칠
예측: 이천백오십칠

en → num: two thousand one hundred fifty seven
정답: 2157
예측: 2157

en → ko: two thousand one hundred fifty seven
정답: 이천백오십칠
예측: 이천백오십칠

ko → num: 이천백오십칠
정답: 2157
예측: 2157

ko → en: 이천백오십칠
정답: two thousand one hundred fifty seven
예측: two thousand one hundred fifty seven



## 7. 모델 저장 및 다운로드

학습한 모델 가중치와 어휘 사전을 `.pt` 파일로 저장합니다.


In [8]:
MODEL_FILE = "Num2Text_LSTM_Greedy.pt"
torch.save({
    "encoder": encoder.state_dict(),
    "decoder": decoder.state_dict(),
    "vocab": vocab,
    "embed_size": EMBED_SIZE,
    "hidden_size": HIDDEN_SIZE,
}, MODEL_FILE)
print("저장 완료:", MODEL_FILE)

# Colab에서 모델 파일을 내려받으려면 주석을 해제합니다.
# from google.colab import files
# files.download(MODEL_FILE)

저장 완료: Num2Text_LSTM_Greedy.pt


## 8. 마지막 셀: 저장한 모델 다시 불러와 테스트

저장된 `.pt` 파일을 불러와 테스트합니다. 실행 결과를 포함한 노트북을 제출합니다.


In [9]:
# 저장한 모델을 다시 불러옵니다.
checkpoint = torch.load("Num2Text_LSTM_Greedy.pt", map_location=device, weights_only=True)
vocab = checkpoint["vocab"]
encoder = Encoder(len(vocab), checkpoint["embed_size"], checkpoint["hidden_size"]).to(device)
decoder = Decoder(len(vocab), checkpoint["embed_size"], checkpoint["hidden_size"]).to(device)
encoder.load_state_dict(checkpoint["encoder"])
decoder.load_state_dict(checkpoint["decoder"])
encoder.eval()
decoder.eval()

# 테스트 데이터에서 6방향 번역 확인
for source, target in [("num", "en"), ("en", "num"),
                       ("num", "ko"), ("ko", "num"),
                       ("en", "ko"), ("ko", "en")]:
    example = next(row for row in test_pairs if row[0] == source and row[1] == target)
    _, _, x, answer = example
    print(f"{source} → {target} | 입력: {x}")
    print("정답:", answer)
    print("결과:", translate(x, source, target))
    print("-" * 40)

# 직접 번역할 때 사용
# print(translate("12345", "num", "en"))
# print(translate("12345", "num", "ko"))
# 각 방향별 30개를 확인합니다. 비교 시 공백은 제외합니다.
print("\n[방향별 테스트 정확도 (30개씩)]")
for source, target in [("num", "en"), ("en", "num"),
                       ("num", "ko"), ("ko", "num"),
                       ("en", "ko"), ("ko", "en")]:
    examples = [p for p in test_pairs if p[0] == source and p[1] == target][:30]
    correct = sum(translate(x, source, target).replace(" ", "") == answer.replace(" ", "")
                  for _, _, x, answer in examples)
    print(f"{source} → {target}: {correct}/{len(examples)}")


num → en | 입력: 2157
정답: two thousand one hundred fifty seven
결과: two thousand one hundred fifty seven
----------------------------------------
en → num | 입력: two thousand one hundred fifty seven
정답: 2157
결과: 2157
----------------------------------------
num → ko | 입력: 2157
정답: 이천백오십칠
결과: 이천백오십칠
----------------------------------------
ko → num | 입력: 이천백오십칠
정답: 2157
결과: 2157
----------------------------------------
en → ko | 입력: two thousand one hundred fifty seven
정답: 이천백오십칠
결과: 이천백오십칠
----------------------------------------
ko → en | 입력: 이천백오십칠
정답: two thousand one hundred fifty seven
결과: two thousand one hundred fifty seven
----------------------------------------

[방향별 테스트 정확도 (30개씩)]
num → en: 30/30
en → num: 30/30
num → ko: 30/30
ko → num: 30/30
en → ko: 30/30
ko → en: 30/30
